# SpaceX Falcon 9 First-Stage Landing Prediction
The final capstone stage treats landing success as a supervised binary-classification problem. Four models are compared: Logistic Regression, Support Vector Machine, Decision Tree and K-Nearest Neighbors.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

DATA_URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv'
FEATURE_URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_3.csv'
data = pd.read_csv(DATA_URL)
X = pd.read_csv(FEATURE_URL)
Y = data['Class'].to_numpy()


## Standardize and split the data
The IBM lab uses an 80/20 train/test split with `random_state=2`. With 90 launch records this creates **72 training samples and 18 test samples**.


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, Y_train, Y_test = train_test_split(
    X_scaled, Y, test_size=0.2, random_state=2, stratify=None
)
print('Training samples:', len(Y_train))
print('Test samples:', len(Y_test))


## Logistic Regression


In [ ]:
lr_grid = {'C':[0.01,0.1,1], 'penalty':['l2'], 'solver':['lbfgs']}
lr_cv = GridSearchCV(LogisticRegression(max_iter=2000), lr_grid, cv=10)
lr_cv.fit(X_train, Y_train)


## Support Vector Machine
The assessed course run selected the **RBF kernel** as the strongest SVM kernel on validation data.


In [ ]:
svm_grid = {
    'kernel':['linear','rbf','sigmoid'],
    'C':np.logspace(-3,3,5),
    'gamma':np.logspace(-3,3,5)
}
svm_cv = GridSearchCV(SVC(), svm_grid, cv=10)
svm_cv.fit(X_train, Y_train)


## Decision Tree
The decision tree is tuned for criterion, splitter, depth, feature count and minimum sample settings. In the course assessment run, the tuned Decision Tree achieved **83.33% test accuracy (15 of 18 test launches classified correctly)**.


In [ ]:
tree_grid = {
    'criterion':['gini','entropy'],
    'splitter':['best','random'],
    'max_depth':[2,4,6,8,10,12,14,16,18],
    'max_features':[None,'sqrt','log2'],
    'min_samples_leaf':[1,2,4],
    'min_samples_split':[2,5,10]
}
tree_cv = GridSearchCV(DecisionTreeClassifier(random_state=2), tree_grid, cv=10)
tree_cv.fit(X_train, Y_train)


## K-Nearest Neighbors


In [ ]:
knn_grid = {
    'n_neighbors':list(range(1,11)),
    'algorithm':['auto','ball_tree','kd_tree','brute'],
    'p':[1,2]
}
knn_cv = GridSearchCV(KNeighborsClassifier(), knn_grid, cv=10)
knn_cv.fit(X_train, Y_train)


## Compare models on unseen test data


In [ ]:
models = {
    'Logistic Regression': lr_cv,
    'SVM': svm_cv,
    'Decision Tree': tree_cv,
    'KNN': knn_cv
}
results = []
for name, model in models.items():
    pred = model.predict(X_test)
    results.append({
        'Model': name,
        'CV Accuracy': model.best_score_,
        'Test Accuracy': accuracy_score(Y_test, pred),
        'Best Parameters': model.best_params_
    })
results_df = pd.DataFrame(results).sort_values('Test Accuracy', ascending=False)
results_df


## Confusion matrix for the selected model
A confusion matrix is essential because overall accuracy alone can hide the balance between correctly identified successful and unsuccessful landings.


In [ ]:
best_name = results_df.iloc[0]['Model']
best_model = models[best_name]
yhat = best_model.predict(X_test)
cm = confusion_matrix(Y_test, yhat)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No success','Success'],
            yticklabels=['No success','Success'])
plt.xlabel('Predicted'); plt.ylabel('Actual');
plt.title(f'Confusion Matrix - {best_name}'); plt.show()
print(classification_report(Y_test, yhat))


## Conclusion
The machine-learning stage shows that first-stage landing success is predictable from structured launch attributes. The assessment run achieved **83.33% test accuracy** with the tuned Decision Tree; the SVM validation search favored an **RBF kernel**. The small 18-launch test set means individual predictions have a noticeable effect on the reported percentage, so cross-validation and confusion-matrix inspection are important alongside accuracy.
